In [5]:
import zipfile
import re
import pandas as pd
from pathlib import Path

import tiktoken

ZIP_PATH = "LCLC_1934-1940_only_unique_text_leaflets.zip"
LOCATION_CSV = "lclc_manual_location_table.csv"  # id,assigned_location

START_YEAR = 1934
END_YEAR = 1939

enc = tiktoken.get_encoding("cl100k_base")


def extract_field(content: str, field: str) -> str | None:
    pattern = rf"^{re.escape(field)}:\s*(.*)$"
    match = re.search(pattern, content, flags=re.MULTILINE)
    return match.group(1).strip() if match else None


def extract_text(content: str) -> str:
    # Ņem tikai saturu pēc "text:"
    parts = re.split(r"^text:\s*$", content, maxsplit=1, flags=re.MULTILINE)
    if len(parts) == 2:
        return parts[1].strip()
    return ""


def extract_year(date_str: str | None) -> int | None:
    if not date_str:
        return None
    match = re.search(r"(19\d{2})", date_str)
    return int(match.group(1)) if match else None


def normalize_id(raw_id: str | None) -> str | None:
    if raw_id is None:
        return None
    return raw_id.strip()


def count_words_latvian(text: str) -> int:
    # Unicode burti; der latviešu tekstam
    words = re.findall(r"\b[^\W\d_]+\b", text, flags=re.UNICODE)
    return len(words)


# Location table
loc_df = pd.read_csv(LOCATION_CSV, dtype={"id": str})
loc_df["id"] = loc_df["id"].astype(str).str.strip()
loc_df["assigned_location"] = loc_df["assigned_location"].astype(str).str.strip()

location_map = dict(zip(loc_df["id"], loc_df["assigned_location"]))

records = []

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    txt_files = [name for name in z.namelist() if name.lower().endswith(".txt")]

    for file_path in txt_files:
        with z.open(file_path) as f:
            content = f.read().decode("utf-8", errors="replace")

        raw_id = normalize_id(extract_field(content, "id"))
        date_str = extract_field(content, "date")
        year = extract_year(date_str)

        if year is None or not (START_YEAR <= year <= END_YEAR):
            continue

        text = extract_text(content)

        if not text:
            continue

        location = location_map.get(raw_id, "UNKNOWN")

        token_count = len(enc.encode(text))
        word_count = count_words_latvian(text)

        records.append({
            "id": raw_id,
            "file_path": file_path,
            "year": year,
            "date": date_str,
            "assigned_location": location,
            "is_riga": location == "Rīga",
            "word_count": word_count,
            "token_count": token_count,
        })

df = pd.DataFrame(records)

summary = (
    df.assign(group=df["is_riga"].map({True: "Rīga", False: "Ārpus Rīgas"}))
      .groupby("group", as_index=False)
      .agg(
          documents=("id", "count"),
          words=("word_count", "sum"),
          tokens=("token_count", "sum")
      )
)

summary["documents_%"] = summary["documents"] / summary["documents"].sum() * 100
summary["words_%"] = summary["words"] / summary["words"].sum() * 100
summary["tokens_%"] = summary["tokens"] / summary["tokens"].sum() * 100

summary = summary.sort_values("group", ascending=False)

print(summary)

# Ja vajag detalizēti pa vietām
location_summary = (
    df.groupby("assigned_location", as_index=False)
      .agg(
          documents=("id", "count"),
          words=("word_count", "sum"),
          tokens=("token_count", "sum")
      )
      .sort_values("documents", ascending=False)
)

location_summary["documents_%"] = location_summary["documents"] / location_summary["documents"].sum() * 100
location_summary["tokens_%"] = location_summary["tokens"] / location_summary["tokens"].sum() * 100

print(location_summary)

# Saglabā rezultātus
summary.to_csv("riga_vs_non_riga_token_summary_1934_1939.csv", index=False)
location_summary.to_csv("location_token_summary_1934_1939.csv", index=False)
df.to_csv("leaflet_token_counts_1934_1939.csv", index=False)

         group  documents   words  tokens  documents_%    words_%   tokens_%
1  Ārpus Rīgas         49   23676   83257    20.081967  16.665845  16.472737
0         Rīga        195  118387  422166    79.918033  83.334155  83.527263
    assigned_location  documents   words  tokens  documents_%   tokens_%
7                Rīga        195  118387  422166    79.918033  83.527263
4   Latgales apgabals         13    6330   22045     5.327869   4.361693
2          Daugavpils         11    5903   20925     4.508197   4.140097
5             Latvija         10    5154   18267     4.098361   3.614200
9           Ventspils          6    2384    8521     2.459016   1.685915
3             Jelgava          4    1589    5440     1.639344   1.076326
0              Bauska          1     295     976     0.409836   0.193106
1                Bēne          1     514    1870     0.409836   0.369987
6   Malienas apgabals          1     191     661     0.409836   0.130782
8               Sloka          1     85